# Fase 2 — Extração de texto + OCR + limpeza

Petições iniciais do STF (ADI/ADC/ADO/ADPF).

Este notebook apenas **executa** o pipeline; toda a lógica está em `src/ocr_pipeline.py`,
para poder rodar também pela linha de comando e ser reaproveitada nas próximas fases.

- **Entrada:** `data/01_raw/pdf/<ID>.pdf` — cópias registradas por `scripts/00_registrar_documentos.py`
- **Saída:** `data/02_processed/ocr/<ID>.json` e `data/02_processed/clean/<ID>.json` (texto por página)
- **Status:** `metadata/documentos.csv` · **log:** `logs/processing.csv`

O passo a passo e a justificativa de cada decisão estão em `docs/ocr_e_limpeza.md`.

## 1. Ambiente

In [ ]:
# !apt-get -qq install tesseract-ocr tesseract-ocr-por
# !pip install -q pytesseract PyMuPDF pandas
# from google.colab import drive; drive.mount('/content/drive')

import sys
from pathlib import Path

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import ocr_pipeline as op

_, msg = op.configurar_tesseract()
print(msg)

## 2. Rodar o pipeline

Processa todo PDF de `data/01_raw/pdf/` que ainda não tem `clean/<ID>.json`. Páginas já
lidas por OCR do mesmo PDF são reaproveitadas; `refazer=True` refaz tudo, inclusive o OCR.

In [ ]:
df = op.processar_pasta(
    refazer=False,
    workers=1,         # >1 processa vários PDFs em paralelo
)
df

## 3. Documentos — status e qualidade de cada um

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 60)

documentos = pd.read_csv(op.CAMINHO_DOCUMENTOS)
documentos[["id", "processo", "relator", "status_ocr", "metodo", "paginas",
            "paginas_ocr", "paginas_ocr_fallback", "conf_ocr_media",
            "chars_clean", "reducao_pct", "linhas_removidas"]]

## 4. Validação visual: antes x depois da limpeza

Confere se a limpeza tirou só ruído (carimbo, assinatura, numeração) e não conteúdo.

In [ ]:
import json

def pagina(etapa, doc_id, n):
    dados = json.loads((op.DIR_PROCESSADO / etapa / (doc_id + ".json")).read_text(encoding="utf-8"))
    p = dados["pages"][n - 1]
    return "[pagina %d | %s | conf %s]\n%s" % (p["page"], p["metodo"], p["conf"], p["text"])

ID = "ADI_5823"
PAGINA = 5

print("=" * 30, "OCR (bruto)", "=" * 30)
print(pagina("ocr", ID, PAGINA)[:1200])
print()
print("=" * 30, "CLEAN", "=" * 30)
print(pagina("clean", ID, PAGINA)[:1200])

## 5. Log de processamento

In [ ]:
pd.read_csv(op.CAMINHO_LOG).tail(20)